# Assignment 8 — Robustness Analysis

**Course:** EPA141A Model-Based Decision Making — Delft University of Technology  
**Model:** JUSTICE  


---

## Learning Outcomes

After completing this assignment you will be able to:

1. Re-evaluate a set of Pareto-optimal policies across an ensemble of climate
   scenarios.

2. Compute and interpret **satisficing scores** — the fraction of scenarios in
   which a policy meets an acceptable threshold on all objectives simultaneously.

3. Compute and interpret **minimax regret** — the worst-case loss a policy
   incurs relative to the best achievable outcome in each scenario.

---

## Assignment Overview

This assignment re-evaluates your Pareto-optimal policies by re-running JUSTICE across multiple climate scenarios (FAIR ensemble members) and applying two complementary robustness metrics:

- **Satisficing** — does the policy meet an acceptable threshold on every objective, across enough scenarios?
- **Minimax regret** — which policy has the smallest worst-case loss relative to what was achievable?

**What you will do**

- Run `run_reeval.py` to evaluate every policy under multiple climate scenarios
- Apply satisficing analysis — choose thresholds, compute scores, and visualise
- Apply minimax regret — compute worst-case performance 

**What you will need**

- Your **reference set** from Assignment 6 (the Pareto-optimal solutions)
- Your **config file** the one you used to run the optimization or optimizations

**What you will produce**

- A heatmap showing which objectives are hardest to satisfy
- A maximum regret ranking of all policies
- A CDF plot of worst-case regret across the Pareto front
- Your recommended solution based on the analysis

> **Scope note:** This is a simplified robustness analysis. A full robustness
> analysis would test policy performance across uncertainty in many parameters
> simultaneously, you can think of all the model assumptions, economic inputs, damage functions, and
> more. Here we vary only the climate uncertainty (FAIR ensemble members),
> holding everything else fixed. This is a meaningful first step, but keep in
> mind that the robustness scores you compute reflect climate uncertainty only,
> not the full uncertainty space the model operates in.

___

## Setup — Imports, paths, and model constants

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.path as _mpath
import seaborn as sns

# ── Patch matplotlib Path deepcopy ────────────────────────────────────────────
def _patched_path_deepcopy(self, memo=None):
    if memo is None: memo = {}
    new_path = _mpath.Path.__new__(_mpath.Path)
    memo[id(self)] = new_path
    verts = self._vertices.copy()
    codes = self._codes.copy() if self._codes is not None else None
    new_path.__init__(verts, codes,
                      _interpolation_steps=self._interpolation_steps, readonly=False)
    return new_path
_mpath.Path.__deepcopy__ = _patched_path_deepcopy

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib; matplotlib.use("Agg")

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

# ── Paths ─────────────────────────────────────────────────────────────────────
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))
_PLOTS_DIR   = os.path.join(_NOTEBOOK_DIR, "plots")
os.makedirs(_PLOTS_DIR, exist_ok=True)

# ── Objectives ────────────────────────────────────────────────────────────────
OBJECTIVES = ["welfare", "years_above_2C", "welfare_loss_damage", "welfare_loss_abatement"]
OBJ_LABELS = ["Welfare", "Yrs > 2°C", "WL Damage", "WL Abatement"]


print(f"Results root : {RESULTS_ROOT}")
print("Setup OK")


## Step 1 — Re-evaluation using the EMA Workbench

To assess the robustness of your Pareto-optimal policies, you need to re-run JUSTICE for every policy under multiple climate scenarios. This is a computationally expensive operation — running it sequentially inside a notebook would take hours on a single core.

Instead, you can adapt the provided script [assignments_ema/run_reeval.py](assignments_ema/run_reeval.py), which runs the experiment in parallel using all available CPU cores and saves the results.

### 1.1. What the script does

The script uses the EMA Workbench (`perform_experiments`) to handle the full factorial experiment — every policy × every scenario.

These are the steps implemented in the script:

1. **Set up paths** — figures out where it lives on disk and builds paths to the JUSTICE model, the config file, and the results folder.

2. **Load your config file** — reads your model settings (start year, end year, timestep, scenario index, etc.) — the same settings used during optimisation.

3. **Compute model constants** — determines how many timesteps and regions the model has, and what shape the RBF parameters take (8 centers, 8 radii, 228 weights).

4. **Define the model wrapper** — wraps JUSTICE in a plain Python function that EMA Workbench calls once per (policy, scenario) pair. It must be at the top level of the script — not inside a notebook cell — so Python's multiprocessing system can distribute it to worker processes. Each call:
   - Receives the 244 RBF parameters and the climate ensemble index as inputs
   - Rebuilds the RBF and runs JUSTICE forward through time (identical to Assignment 7)
   - Returns the four objectives: welfare, years above 2°C, welfare loss from damage, and welfare loss from abatement

5. **Load the reference set** — reads your reference set CSV, and identifies the 244 lever columns (this is essentially what you need for the reruns).

6. **Check if results already exist** — if the output files are already on disk, the script exits immediately so you don't accidentally re-run a long computation.

7. **Register the model with EMA Workbench** — creates a `Model` object and declares:
   - **Uncertainties** — `climate_ensemble_index` (integer 1–1000): which FAIR climate trajectory to use. Varies across scenarios.
   - **Levers** — the 244 RBF parameters (centers, radii, weights). Vary across policies, fixed within each policy.
   - **Outcomes** — the four scalar objectives.

> **Note on objective directions:** In Assignment 4, `welfare_loss_damage` and
> `welfare_loss_abatement` were declared as `MAXIMIZE` during optimisation — this
> was an EMA Workbench convention for handling positive quantities, not a sign
> that higher losses are better. Here, with no optimiser involved, they are
> correctly declared as `MINIMIZE`: lower welfare loss is always better.

8. **Build policy and scenario lists** — creates one `Policy` object per row in the reference set and one `Scenario` object per selected FAIR ensemble member.

9. **Run in parallel** — distributes all (policy, scenario) combinations across CPU cores using `MultiprocessingEvaluator`.

10. **Reshape and save** — reorganises the results into two files:
    - **`<run_name>_<n_policies>p_<n_scenarios>s.npy`** — a 3D NumPy array of shape `(n_policies, n_scenarios, n_objectives)` with the four objective values for every combination. This is what the notebook uses for robustness analysis.
    - **`<run_name>_<n_policies>p_<n_scenarios>s_experiments.csv`** — a flat table recording the inputs (policy, scenario, lever values) for every run.


### 1.2. Running the script

**Step 1 — Update the script**

Open [assignments_ema/run_reeval.py](assignments_ema/run_reeval.py) and make the necessary changes, at the very least, you'll need to:

1. Replace `config_ssp245.json` with your own config
2. Replace `reference_set_utilitarian.csv` with your reference set.

**Step 2 — Run from the terminal**

Open a terminal in the `epa141a` folder and run:

```bash
# Quick test — verify everything works before a full run
.venv/bin/python assignments_ema/run_reeval.py --n_scenarios 10

# Full run (this will take several hours)
.venv/bin/python assignments_ema/run_reeval.py --n_scenarios 1000

# Limit cores if needed (e.g. leave one free for other work)
.venv/bin/python assignments_ema/run_reeval.py --n_scenarios 1000 --n_cores 4
```

> Tip: start running the script as is with a smoke test i.e. `--n_scenarios 5` to confirm your config and reference set paths are correct before implementing further changes and committing to a full run, which can take foreverr. 




In [ ]:
# ── Load reference set ────────────────────────────────────────────────────────
REF_SET_PATH = os.path.join(
    RESULTS_ROOT, "reference_set_CONSUMPTION_CORRIDOR_PRIORITARIAN.csv"
)

if not os.path.exists(REF_SET_PATH):
    raise FileNotFoundError(
        f"Reference set not found: {REF_SET_PATH}\n"
        "Run Assignment 6 (Steps 1–3) first to build and save the reference set."
    )

ref_set = pd.read_csv(REF_SET_PATH)

# The CSV contains duplicate columns: underscore ("center_0") and space
# ("center 0") versions of every parameter column.  The model wrapper expects
# space-named levers, so drop the underscore-named duplicates.
_underscore_params = [
    c for c in ref_set.columns
    if any(c.startswith(p) for p in ["center_", "radii_", "weights_"])
]
ref_set = ref_set.drop(columns=_underscore_params)
ref_set = ref_set[ref_set["welfare"] < 1e5].reset_index(drop=True)

OPT_OBJECTIVES = ["welfare", "fraction_above_threshold",
                  "welfare_loss_damage", "welfare_loss_abatement"]
LEVER_COLS = [c for c in ref_set.columns if c not in OPT_OBJECTIVES]

print(f"Reference set: {REF_SET_PATH}")
print(f"  {len(ref_set)} policies  |  {len(LEVER_COLS)} lever columns")
print(f"  Lever columns (first 5): {LEVER_COLS[:5]}")
print()
print("Optimisation objective ranges:")
print(ref_set[OPT_OBJECTIVES].describe().round(3).to_string())

In [ ]:
import json, sys

# ── JUSTICE path ──────────────────────────────────────────────────────────────
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))
if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)
os.chdir(_JUSTICE_ROOT)

from justice.model import JUSTICE
from justice.util.data_loader import DataLoader
from justice.util.enumerations import (
    Abatement, DamageFunction, Economy, WelfareFunction
)
from justice.util.emission_control_constraint import EmissionControlConstraint
from justice.util.model_time import TimeHorizon
from justice.objectives.objective_functions import years_above_temperature_threshold
from solvers.emodps.rbf import RBF
from ema_workbench import (
    Model, RealParameter, IntegerParameter, ScalarOutcome, ema_logging,
)

ema_logging.log_to_stderr(ema_logging.INFO)

# ── Model constants from config ───────────────────────────────────────────────
with open(os.path.join(_NOTEBOOK_DIR, "../config/config_student.json")) as fh:
    _cfg = json.load(fh)

_time_horizon = TimeHorizon(
    start_year=_cfg["start_year"],
    end_year=_cfg["end_year"],
    data_timestep=_cfg["data_timestep"],
    timestep=_cfg["timestep"],
)
N_TIMESTEPS = len(_time_horizon.model_time_horizon)
N_REGIONS   = len(DataLoader().REGION_LIST)
N_INPUTS    = _cfg["n_inputs"]
N_RBFS      = _cfg["n_inputs"] + 2
SCENARIO    = _cfg["reference_ssp_rcp_scenario_index"]
EC_START_TS = _time_horizon.year_to_timestep(
    year=_cfg["emission_control_start_year"],
    timestep=_cfg["timestep"],
)
_MAX_TEMP, _MIN_TEMP = 16.0, 0.0
_MAX_DIFF, _MIN_DIFF = 2.0, 0.0

_rbf_dummy = RBF(n_rbfs=N_RBFS, n_inputs=N_INPUTS, n_outputs=N_REGIONS)
C_SHAPE, R_SHAPE, W_SHAPE = _rbf_dummy.get_shape()

print(f"N_TIMESTEPS={N_TIMESTEPS}, N_REGIONS={N_REGIONS}, N_RBFS={N_RBFS}, N_INPUTS={N_INPUTS}")
print(f"C_SHAPE={C_SHAPE}, R_SHAPE={R_SHAPE}, W_SHAPE={W_SHAPE}")

# ── Model wrapper ─────────────────────────────────────────────────────────────
def model_wrapper_reeval(**kwargs) -> tuple:
    ensemble_index = int(kwargs.pop("climate_ensemble_index"))

    rbf = RBF(n_rbfs=N_RBFS, n_inputs=N_INPUTS, n_outputs=N_REGIONS)
    centers = np.array([kwargs.pop(f"center {i}") for i in range(C_SHAPE[0])])
    radii   = np.array([kwargs.pop(f"radii {i}")  for i in range(R_SHAPE[0])])
    weights = np.array([kwargs.pop(f"weights {i}") for i in range(W_SHAPE[0])])
    rbf.set_decision_vars(np.concatenate([centers, radii, weights]))

    constraint = EmissionControlConstraint(
        max_annual_growth_rate=0.04,
        emission_control_start_timestep=EC_START_TS,
        min_emission_control_rate=0.01,
    )

    model = JUSTICE(
        scenario=SCENARIO,
        climate_ensembles=[ensemble_index],
        economy_type=Economy.NEOCLASSICAL,
        damage_function_type=DamageFunction.KALKUHL,
        abatement_type=Abatement.ENERDATA,
        social_welfare_function_type=WelfareFunction.CONSUMPTION_CORRIDOR_PRIORITARIAN.value[0],
    )
    no_ens = model.no_of_ensembles

    ecr             = np.zeros((N_REGIONS, N_TIMESTEPS, no_ens))
    constrained_ecr = np.zeros_like(ecr)
    prev_temp = np.zeros(no_ens)
    diff      = np.zeros(no_ens)

    for t in range(N_TIMESTEPS):
        constrained_ecr[:, t, :] = constraint.constrain_emission_control_rate(
            ecr[:, t, :], t, allow_fallback=False
        )
        model.stepwise_run(
            emission_control_rate=constrained_ecr[:, t, :],
            timestep=t,
            endogenous_savings_rate=True,
        )
        data_t = model.stepwise_evaluate(timestep=t)
        temp = data_t["global_temperature"][t, :]

        if t % 5 == 0:
            diff      = temp - prev_temp
            prev_temp = temp.copy()

        scaled_temp = (temp - _MIN_TEMP) / (_MAX_TEMP - _MIN_TEMP)
        scaled_diff = (diff - _MIN_DIFF) / (_MAX_DIFF - _MIN_DIFF)

        if t < N_TIMESTEPS - 1:
            ecr[:, t + 1, :] = rbf.apply_rbfs(np.array([scaled_temp, scaled_diff]))

    data = model.evaluate()

    welfare_val = float(np.abs(data["welfare"]))
    welfare_val = welfare_val if np.isfinite(welfare_val) else 1e6

    yrs_above = float(
        years_above_temperature_threshold(data["global_temperature"], threshold=2.0)
    )

    _, _, _, wl_damage = model.welfare_function.calculate_welfare(
        data["damage_cost_per_capita"], welfare_loss=True
    )
    wl_damage = float(np.abs(wl_damage)) if np.isfinite(wl_damage) else 1e6

    _, _, _, wl_abatement = model.welfare_function.calculate_welfare(
        data["abatement_cost_per_capita"], welfare_loss=True
    )
    wl_abatement = float(np.abs(wl_abatement)) if np.isfinite(wl_abatement) else 1e6

    return (welfare_val, yrs_above, wl_damage, wl_abatement)


# ── EMA model definition ──────────────────────────────────────────────────────
ema_model = Model("JUSTICEreeval", function=model_wrapper_reeval)

ema_model.uncertainties = [IntegerParameter("climate_ensemble_index", 1, 1000)]

n_cr = C_SHAPE[0]
n_w  = W_SHAPE[0]
ema_model.levers = (
    [RealParameter(f"center {i}", -1.0, 1.0) for i in range(n_cr)]
    + [RealParameter(f"radii {i}",  0.0, 1.0) for i in range(n_cr)]
    + [RealParameter(f"weights {i}", 0.0, 1.0) for i in range(n_w)]
)

ema_model.outcomes = [
    ScalarOutcome("welfare",                kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("years_above_2C",         kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("welfare_loss_damage",    kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("welfare_loss_abatement", kind=ScalarOutcome.MINIMIZE),
]

print(f"EMA model OK: {len(ema_model.uncertainties)} uncertainty, "
      f"{len(ema_model.levers)} levers ({n_cr} centers + {n_cr} radii + {n_w} weights), "
      f"{len(ema_model.outcomes)} outcomes")

In [ ]:
# ── Scenario and policy setup + re-evaluation run ─────────────────────────────
# Set N_SCENARIOS=5 for a smoke test (verify paths, ~5 min sequential).
# Increase to 50 for a meaningful robustness run, or use run_reeval.py for
# the full 1000-scenario parallel run.

from ema_workbench import Sample, SequentialEvaluator, perform_experiments

N_SCENARIOS      = 5        # ← change to 50 for full notebook run
SCENARIO_INDICES = list(np.linspace(1, 1000, N_SCENARIOS, dtype=int))

N_POLICIES   = len(ref_set)
N_OBJ        = len(OBJECTIVES)

RESULTS_PATH     = os.path.join(
    RESULTS_ROOT, f"reeval_cc_prioritarian_{N_POLICIES}p_{N_SCENARIOS}s.npy"
)
EXPERIMENTS_PATH = os.path.join(
    RESULTS_ROOT, f"reeval_cc_prioritarian_{N_POLICIES}p_{N_SCENARIOS}s_experiments.csv"
)

if os.path.exists(RESULTS_PATH):
    print(f"Results already on disk: {RESULTS_PATH}")
    print("Delete the file to re-run.")
else:
    # One Policy per Pareto solution (levers fixed, climate index varies across scenarios)
    policies = [
        Sample(f"P{pi}", **{col: float(ref_set.iloc[pi][col]) for col in LEVER_COLS})
        for pi in range(N_POLICIES)
    ]

    # One Scenario per FAIR ensemble member
    scenarios = [
        Sample(f"FAIR_{idx}", climate_ensemble_index=int(idx))
        for idx in SCENARIO_INDICES
    ]

    print(f"Policies  : {N_POLICIES}")
    print(f"Scenarios : {N_SCENARIOS}  (indices: {SCENARIO_INDICES})")
    print("Running re-evaluation …")

    with SequentialEvaluator(ema_model) as evaluator:
        experiments_df, outcomes = evaluator.perform_experiments(
            policies=policies, scenarios=scenarios
        )

    # Reshape to (n_policies, n_scenarios, n_objectives)
    raw = np.column_stack([outcomes[o] for o in OBJECTIVES])
    results_arr = raw.reshape(N_POLICIES, N_SCENARIOS, N_OBJ)

    np.save(RESULTS_PATH, results_arr)
    experiments_df.to_csv(EXPERIMENTS_PATH, index=False)
    print(f"Saved: {RESULTS_PATH}")
    print(f"Saved: {EXPERIMENTS_PATH}")

print(f"\nResults shape: ({N_POLICIES}, {N_SCENARIOS}, {N_OBJ})")

In [ ]:
# ── Load re-evaluation results for robustness analysis ────────────────────────
# Run the smoke-test cell below first; for the full run use run_reeval.py.

_results_npy = os.path.join(
    RESULTS_ROOT,
    f"reeval_cc_prioritarian_{len(ref_set)}p_{N_SCENARIOS}s.npy"
)

if not os.path.exists(_results_npy):
    raise FileNotFoundError(
        f"Results not found: {_results_npy}\n"
        "Run the smoke-test cell above (or run_reeval.py for the full run), "
        "then re-execute this cell."
    )

# Shape: (n_policies, n_scenarios, n_objectives)
results_array = np.load(_results_npy)
print(f"Results loaded: shape {results_array.shape}")
print(f"  axes: (policies={results_array.shape[0]}, "
      f"scenarios={results_array.shape[1]}, objectives={results_array.shape[2]})")
print(f"  objectives: {OBJECTIVES}")


## Step 2 — Satisficing analysis

A policy **satisfices** in a scenario if it meets an acceptable threshold on
**every** objective simultaneously. The **satisficing score** is the fraction
of scenarios in which a policy does this.

**Task 1. Choose your thresholds**

For each of the four objectives, decide what counts as "acceptable" performance.
You must define one threshold per objective and justify your choice. Consider:

- Is there an external standard you can use? For example, the Paris Agreement
  temperature target, which could directly inform your threshold?
- If no external standard exists, you can establish a specific percentile across all policies and scenarios.
- Should some objectives be held to a stricter standard than others? Explain why.

Document your reasoning for each threshold before moving on.


**Task 2. Compute satisficing scores**

A policy *satisfices* in a given scenario if it meets **all four thresholds simultaneously**.

To organize this informtion, you could first populate a boolean array of shape `(n_policies, n_scenarios)` where each entry
is `True` if the policy meets all thresholds in that scenario, and `False` otherwise. Where the columns are the scenarios, and the rows are the policies. So, something like this:
|     | S1    | S2    | S3    | S4    |
|-----|-------|-------|-------|-------|
| P0  | True  | True  | False | True  |
| P1  | False | False | False | False |
| P2  | True  | True  | True  | True  |
| P3  | True  | False | True  | False |
| P4  | False | True  | False | True  |


**Task 3. Compute and report satisficing scores**

For each policy, compute its **satisficing score**: the fraction of scenarios in
which it satisfices all objectives. A score of 1.0 means the policy meets all
thresholds in every scenario; a score of 0.0 means it never does.

Report the following:
- The mean and maximum satisficing score across all policies
- How many policies have a satisficing score of zero (never satisfice)
- The per-objective satisficing rate: for each objective separately, what
  fraction of (policy, scenario) pairs meet that threshold? This tells you
  which objective is the hardest constraint to satisfy.

**Task 4. Visualizing satisfying scores**
Create a heatmap with policies as rows (sorted from highest to lowest
satisficing score) and objectives as columns. Each cell shows the fraction
of scenarios in which that policy meets the threshold for that objective.
**Useful references:**
- [`sns.heatmap()` documentation](https://seaborn.pydata.org/generated/seaborn.heatmap.html) — pay attention to `vmin`, `vmax`, `cmap`, `annot`, and `linewidths`
- [`np.argsort()`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) — for sorting row indices by satisficing score

> Tip 1: Use a shared color scale with `vmin=0, vmax=1` so all objectives
are comparable.

> Tip 2: If you decide that dark = meets threshold in most scenarios, then the column with the lighter cells would be the hardest to satisfy.


In [ ]:
# ── Compute per-objective thresholds ─────────────────────────────────────────
# results shape: (n_policies, n_scenarios, n_objectives)
# Flatten to (n_policies * n_scenarios, n_objectives)
flat = results.reshape(-1, N_OBJECTIVES)

# years_above_2C: 25th percentile — stricter, Paris-Agreement-consistent
# All other objectives: median — symmetric, data-driven, no external normative anchor
THRESHOLD_PERCENTILES = [50, 25, 50, 50]   # welfare, years_above_2C, wl_damage, wl_abatement
thresholds = np.array([
    np.nanpercentile(flat[:, i], THRESHOLD_PERCENTILES[i])
    for i in range(N_OBJECTIVES)
])  # shape (n_objectives,)

print("Satisficing thresholds:")
for name, pct, thr in zip(OBJECTIVES, THRESHOLD_PERCENTILES, thresholds):
    print(f"  {name:<30s}: {thr:.3f}  ({pct}th percentile)")

# ── Per (policy, scenario): satisfices all objectives? ────────────────────────
# All objectives are MINIMIZE: satisfices if value <= threshold
satisfices = np.all(results <= thresholds[np.newaxis, np.newaxis, :], axis=2)
# satisfices shape: (n_policies, n_scenarios) — boolean

# ── Satisficing score per policy: fraction of scenarios ───────────────────────
sat_score = np.nanmean(satisfices, axis=1)   # shape (n_policies,)

print(f"\nSatisficing scores across {N_POLICIES} policies:")
print(f"  Mean score  : {sat_score.mean():.1%}")
print(f"  Max score   : {sat_score.max():.1%}  (policy #{sat_score.argmax()})")
print(f"  Zero score  : {(sat_score == 0).sum()} policies never satisfice")

print("\nPer-objective satisficing rate (fraction of policy×scenario pairs meeting threshold):")
for name, thr in zip(OBJECTIVES, thresholds):
    rate = np.nanmean(results[..., OBJECTIVES.index(name)] <= thr)
    print(f"  {name:<30s}: {rate:.1%}")

## Step 3 — Minimax Regret

Satisficing tells you whether a policy is *acceptable*. Minimax regret tells
you which policy has the **smallest worst-case loss** — the one that never
performs catastrophically relative to what was achievable in that scenario.

**Task 1. Compute the per-scenario ideal and anti-ideal.**
For each scenario, find the best and worst value any policy achieves on each
objective. These define the range of performance possible in that scenario.

**Task 2. Compute normalised regret.**
For each (policy, scenario, objective), regret is how far that policy
falls short of the best achievable outcome in that scenario, expressed as a
fraction of the full range. (You can subtract the ideal from each policy's result, then divide by the range (anti-ideal − ideal) computed in Task 1). A regret of 0 means the policy matched the best
possible outcome; a regret of 1 means it achieved the worst possible outcome.

**Task 3. Compute maximum regret per policy.**
For each policy, take the worst-case total regret across all scenarios. This
is the **maximum regret** — the single number that summarises how badly the
policy could perform in the worst climate future.

**Task 4. Identify the minimax-regret policy.**
Sort policies by maximum regret. The policy with the lowest maximum regret is
the most robust choice under this criterion.

**Task 5. Plot a CDF of maximum regret**
Plot the cumulative distribution function of maximum regret across all
policies (x = maximum regret, y = fraction of policies with regret ≤ x).
Mark the minimax-regret policy with a vertical line.

This plot shows the full spread of robustness across the Pareto front —
whether most policies are similarly robust or whether there is a wide range.


In [ ]:
#YOUR CODE HERE

## Reflection Questions

**How does the minimax-regret policy compares to the best satisficing policy? are they the same policy or different ones?**

**What does this tell you about your selected robustness method?**

**Which single policy would you recommend, and why?**
